# Lab 7: DuckDB in the Cloud with MotherDuck - SOLUTION

This notebook contains the solution for Lab 7 exercises.

## 💻 Exercise 1: MotherDuck Connection Setup - SOLUTION

### Solution

In [ ]:
import duckdb
import os

# Solution framework for MotherDuck connection
# Replace with your actual token when ready

# Step 1: Set up environment variables
# os.environ['MOTHERDUCK_TOKEN'] = 'your_actual_token_here'

# Step 2: Connect to MotherDuck
def connect_to_motherduck(database_name='my_database'):
    """Connect to MotherDuck with error handling"""
    try:
        # This will work when you have a valid token
        con = duckdb.connect(f'md:{database_name}')
        print(f"✓ Successfully connected to MotherDuck database: {database_name}")
        return con
    except Exception as e:
        print(f"Connection failed: {e}")
        print("Make sure MOTHERDUCK_TOKEN environment variable is set")
        return None

# Step 3: Test the connection
def test_connection(con):
    """Test the MotherDuck connection"""
    if con is None:
        return False
    
    try:
        # Test basic query
        result = con.execute("SELECT current_database()").fetchone()
        print(f"✓ Current database: {result[0]}")
        
        # List tables
        tables = con.execute("SHOW TABLES").fetchall()
        print(f"✓ Found {len(tables)} tables")
        
        return True
    except Exception as e:
        print(f"Test failed: {e}")
        return False

# Step 4: Create a test database
def create_test_database(con):
    """Create a test table in MotherDuck"""
    if con is None:
        return False
    
    try:
        con.execute("""
            CREATE TABLE test_table AS
            SELECT * FROM range(100)
        """)
        count = con.execute("SELECT COUNT(*) FROM test_table").fetchone()[0]
        print(f"✓ Created test table with {count} rows")
        return True
    except Exception as e:
        print(f"Table creation failed: {e}")
        return False

# Execute the solution (requires actual token)
print("MotherDuck Connection Solution:")
print("="*50)
print("Note: This solution requires actual MotherDuck credentials")
print("1. Sign up at https://motherduck.com")
print("2. Get your service token from settings")
print("3. Set MOTHERDUCK_TOKEN environment variable")
print("4. Run the connection functions above")

## 💻 Exercise 2: Data Upload and Management - SOLUTION

### Solution

In [ ]:
import duckdb

# Solution for data upload and management

def upload_to_motherduck(local_db_path, motherduck_db):
    """Upload local database to MotherDuck"""
    try:
        # Connect to local database
        local_con = duckdb.connect(local_db_path)
        
        # Connect to MotherDuck
        motherduck_con = duckdb.connect(f'md:{motherduck_db}')
        
        # Get list of tables from local database
        tables = local_con.execute("SHOW TABLES").fetchall()
        
        print(f"Found {len(tables)} tables to upload")
        
        # Upload each table
        for table in tables:
            table_name = table[0]
            print(f"Uploading {table_name}...")
            
            # Create table in MotherDuck
            motherduck_con.execute(f"""
                CREATE TABLE {table_name} AS
                SELECT * FROM local_db.{table_name}
            """)
            
            count = motherduck_con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            print(f"  ✓ Uploaded {count} rows")
        
        local_con.close()
        motherduck_con.close()
        
        print("✓ Upload complete")
        return True
        
    except Exception as e:
        print(f"Upload failed: {e}")
        return False

def verify_data_integrity(local_db_path, motherduck_db):
    """Verify data integrity between local and MotherDuck"""
    try:
        local_con = duckdb.connect(local_db_path)
        motherduck_con = duckdb.connect(f'md:{motherduck_db}')
        
        tables = local_con.execute("SHOW TABLES").fetchall()
        
        print("\nVerifying data integrity:")
        for table in tables:
            table_name = table[0]
            
            local_count = local_con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            motherduck_count = motherduck_con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            
            if local_count == motherduck_count:
                print(f"  ✓ {table_name}: {local_count} == {motherduck_count}")
            else:
                print(f"  ✗ {table_name}: {local_count} != {motherduck_count}")
        
        local_con.close()
        motherduck_con.close()
        
        return True
        
    except Exception as e:
        print(f"Verification failed: {e}")
        return False

# Execute the solution (requires actual MotherDuck connection)
print("Data Upload and Management Solution:")
print("="*50)
print("Functions provided for:")
print("1. upload_to_motherduck(local_db_path, motherduck_db)")
print("2. verify_data_integrity(local_db_path, motherduck_db)")
print("\nUsage example:")
print("upload_to_motherduck('data/duckdb_practice.db', 'my_cloud_db')")
print("verify_data_integrity('data/duckdb_practice.db', 'my_cloud_db')")